# 00 · Customer Potential del cliente (solo CP): validación y PDV de la ZM

Desde 2026-10-01 el 05 y el 06 usan **solo el CP** ("ya no ocupamos ventas, ahora solo ocupamos CP"): la venta de cada tienda es la
**venta media de la categoría** que trae el CP (`PotentialQuantitative_<cat>`, `config.CP_CATEGORIA`). Este notebook reemplaza al
00 de ventas cuando el cliente no entrega archivo de ventas (lo elige `src/correr.py`) y deja la **misma tabla de PDV** que usan el
04, el 05 y el 06 (`pdv_<cliente>_<zm>.parquet`).

* **Insumos obligatorios** (sin ellos no se avanza; se piden al cliente): el CP (`data/raw/<ciudad>/<cliente>/cp/`) y AltScore
  (`enrichedgeodata/geohex_geodig.parquet`). Se registra su huella (filas, fecha, SHA-256).
* **Coordenadas:** escala perdida, lat/lon invertidas y signo; se quedan las que caen en un municipio del estado (polígono del
  Marco Geoestadístico; no se confía en `pos_city`).
* **ZM:** municipios de la delimitación SEDATU-CONAPO-INEGI (`config.ZM_MUNICIPIOS`).
* **Canal:** `src/cadenas.py` (misma regla que el DENUE) sobre el nombre del PDV.
* **Venta:** con venta = venta media de la categoría > 0. Las métricas de actividad (meses, recencia) no existen sin ventas: quedan
  vacías y la venta "de vida" del 04 es la venta media del CP.

In [1]:
import hashlib
import os
import sys
from datetime import datetime
from pathlib import Path

os.environ.setdefault("CIUDAD", "merida")
BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(BASE / "src"))

import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display

import config as C
import cadenas
import eda
from descargas import descargar_fuente, extraer

assert C.CIUDAD == C.CLIENTE_CIUDAD, f"No hay cliente configurado para {C.CIUDAD} (config.CLIENTES)"
pd.set_option("display.width", 220, "display.max_columns", 40, "display.float_format", "{:,.3f}".format)
CAT = C.CP_CATEGORIA
print(f"Cliente: {C.CLIENTE_NOMBRE} · {C.ZM_NOMBRE} · categoría del CP: {CAT}")

# --- requisitos: CP y AltScore son OBLIGATORIOS (usuario): si falta uno, se detiene y se pide al cliente ---
ALT = sorted(Path(C.CLIENTE_ALTSCORE).glob("*.parquet")) if Path(C.CLIENTE_ALTSCORE).exists() else []
assert Path(C.CLIENTE_CP).exists(), f"Falta el CP del cliente: {C.CLIENTE_CP}. Pedirlo; no se puede avanzar sin él."
assert ALT, f"Falta AltScore del cliente en {C.CLIENTE_ALTSCORE} (geohex_geodig.parquet). Pedirlo; no se puede avanzar sin él."
FUENTES_CLIENTE = pd.DataFrame([{
    "insumo": "Customer Potential" if Path(f) == Path(C.CLIENTE_CP) else "AltScore", "archivo": Path(f).name,
    "ruta": str(Path(f).relative_to(BASE)), "bytes": Path(f).stat().st_size,
    "modificado": datetime.fromtimestamp(Path(f).stat().st_mtime).strftime("%Y-%m-%d %H:%M"),
    "sha256": hashlib.sha256(Path(f).read_bytes()).hexdigest()} for f in [C.CLIENTE_CP, *ALT]])
FUENTES_CLIENTE.to_csv(C.PROC / f"fuentes_cliente_{C.CLIENTE}_00.csv", index=False)
display(FUENTES_CLIENTE)
descargar_fuente(C.FUENTES["mg"])
D_MG = extraer(C.ARCHIVOS["mg"])

Cliente: Arca · ZM Guadalajara · categoría del CP: CustomCat_sueros


,insumo,archivo,ruta,bytes,modificado,sha256
0,Customer Potential,cp_arca_mex_sueros_202609.csv,data\raw\guadalajara\arca\cp\cp_arca_mex_suero...,55795818,2026-10-05 14:14,643f7cd6408e52bcad7194dd74743f63e5ddbad275621c...
1,AltScore,geohex_geodig.parquet,data\raw\guadalajara\arca\enrichedgeodata\geoh...,140202613,2026-10-05 14:16,13c4b4bddddaacf3f99c13ae2494dad601ed3e66b89197...


[ok] mg2025eic_14_jalisco.zip (135.7 MB)


## 1. Carga e integridad

In [2]:
cp = pd.read_csv(C.CLIENTE_CP)
FUENTES_CLIENTE.loc[FUENTES_CLIENTE.insumo == "Customer Potential", "filas"] = len(cp)
falt = [f"{p}_{CAT}" for p in ("PotentialQuantitative", "PotentialQuantitativeFinal", "PotentialQualitative", "PotentialEstimatedToCover")
        if f"{p}_{CAT}" not in cp.columns]
assert not falt, f"El CP no trae las columnas de la categoría {CAT}: {falt}"
assert cp.pos_id.is_unique, "pos_id repetido en el CP"
cp[["cadena", "canal"]] = cadenas.clasificar(cp.pos_name)
print(f"CP: {len(cp):,} PDV · {cp.pos_city.nunique():,} ciudades del CP · columnas {len(cp.columns)}")

CP: 262,029 PDV · 43 ciudades del CP · columnas 24


## 2. Coordenadas y filtro a la ZM

Reglas: (1) escala: si |valor| > 1000 se divide hasta quedar con dos dígitos enteros; (2) si |lat| > 80 y |lon| < 30 se
intercambian; (3) lat positiva, lon negativa (México); (4) el punto debe caer en un municipio del estado.

In [3]:
def escala(v):
    v = v.astype(float).copy()
    g = v.abs() > 1000
    v[g] = v[g] / 10 ** (np.floor(np.log10(v[g].abs())) - 1)
    return v

lat, lon = escala(cp.pos_latitude), escala(cp.pos_longitude)
inv = (lat.abs() > 80) & (lon.abs() < 30)
lat, lon = lat.where(~inv, lon), lon.where(~inv, lat)
lat, lon = lat.abs(), -lon.abs()
cambio = ~(np.isclose(lat, cp.pos_latitude) & np.isclose(lon, cp.pos_longitude))
mun = gpd.read_file(next(D_MG.rglob(f"{C.ENT}mun.shp")))[["CVE_MUN", "NOMGEO", "geometry"]]
pts = gpd.GeoDataFrame(cp[["pos_id"]], geometry=gpd.points_from_xy(lon, lat), crs=4326).to_crs(mun.crs)
j = gpd.sjoin(pts, mun, how="left", predicate="within")
j = j[~j.index.duplicated()]
cp["cve_mun"], cp["municipio_mg"] = j.CVE_MUN.reindex(cp.index), j.NOMGEO.reindex(cp.index)
en_estado = cp.cve_mun.notna()
cp["lat"], cp["lon"] = lat.where(en_estado), lon.where(en_estado)
cp["coord_estado"] = np.select([en_estado & ~cambio, en_estado], ["original", "corregida"], f"fuera de {C.NOM_ENT}")
cp["en_zm"] = cp.cve_mun.isin(C.ZM_MUNICIPIOS)
coords = cp.coord_estado.value_counts().to_frame("PDV")
display(coords)
cp = cp[cp.en_zm].copy()                                # desde aquí, universo = CP de la ZM
print(f"PDV del CP en {C.ZM_NOMBRE}: {len(cp):,} (municipios: {', '.join(C.ZM_MUNICIPIOS.values())})")
dup = cp.duplicated(["lat", "lon"], keep=False)
print(f"PDV de la ZM con coordenadas idénticas a otro PDV: {dup.sum():,} (dobles registros o geocodificación por calle)")

,PDV
coord_estado,
fuera de Jalisco,202842
original,59187


PDV del CP en ZM Guadalajara: 39,236 (municipios: Acatlán de Juárez, Guadalajara, Ixtlahuacán de los Membrillos, Juanacatlán, El Salto, Tlajomulco de Zúñiga, San Pedro Tlaquepaque, Tonalá, Zapopan, Zapotlanejo)
PDV de la ZM con coordenadas idénticas a otro PDV: 5,833 (dobles registros o geocodificación por calle)


## 3. Venta y potencial de la categoría (CP)

In [4]:
V = cp[f"PotentialQuantitative_{CAT}"]
cp["con_venta"] = V.gt(0)
por_mun = cp.groupby("municipio_mg").agg(PDV=("pos_id", "size"), con_venta=("con_venta", "sum"), venta=(f"PotentialQuantitative_{CAT}", "sum"))
por_mun["% con venta"] = por_mun.con_venta / por_mun.PDV * 100
por_sub = cp.groupby(["canal", "pos_subchannel"]).agg(PDV=("pos_id", "size"), con_venta=("con_venta", "sum"),
                                                     venta_media=(f"PotentialQuantitative_{CAT}", "median")).sort_values("PDV", ascending=False)
clase = pd.crosstab(cp.canal, cp[f"PotentialQualitative_{CAT}"].fillna("sin venta"))
vacios = cp.loc[~cp.con_venta, [f"PotentialQuantitativeFinal_{CAT}", f"PotentialQualitative_{CAT}"]].isna().mean() * 100
display(por_mun.round(1)); display(por_sub.head(20).round(2)); display(clase)
x = V[V > 0]
uni = pd.DataFrame({"PDV con venta": [len(x)], "mediana cajas/mes": [x.median()], "p90": [x.quantile(.9)], "media": [x.mean()],
                    "Gini": [eda.gini(x.to_numpy()) if hasattr(eda, "gini") else np.nan]}).round(3)
display(uni)

,PDV,con_venta,venta,% con venta
municipio_mg,,,,
Acatlán de Juárez,237,175,"1,158.600",73.800
El Salto,2361,1391,"6,628.500",58.900
Guadalajara,10571,6444,"22,452.700",61.000
Ixtlahuacán de los Membrillos,493,413,"1,948.300",83.800
Juanacatlán,299,192,981.900,64.200
San Pedro Tlaquepaque,5594,3391,"12,507.000",60.600
Tlajomulco de Zúñiga,4483,3216,"14,348.100",71.700
Tonalá,5100,3086,"11,614.400",60.500
Zapopan,9323,5695,"26,216.100",61.100


PDV  con_venta  venta_media
canal       pos_subchannel                                                              
Tradicional Abarrotes / Almacenes / Bodegas / Víveres      15861       7980        4.340
            Cerveza y Licores                               7767       6009        2.700
            Hogar con Venta                                 7048       4399        1.600
            Estanquillos / kioscos                          4194       3276        3.500
            Frutas y Verduras                               1450       1021        2.100
            Tiendas de Alimentos Especializados Orgánicos   1029        685        1.590
            Farmacia Independiente                           671        439        1.200
            Carnicería / Pollería / Pescadería               529        369        1.760
            Minisuper / Minimarket                           377        304        5.390
            Panadería / Pastelería                           141         81        1.170
            Tortillería                                       97         53        1.290
Moderno     Cerveza y Licores                                 28         17        2.250
Tradicional TDC/Proximidad Independiente                      22         16        4.460
Moderno     Abarrotes / Almacenes / Bodegas / Víveres         16          7        6.490
            Farmacia Independiente                             3          1        0.870
            Frutas y Verduras                                  2          2        1.060
            Estanquillos / kioscos                             1          1       12.310

PotentialQualitative_CustomCat_sueros,High,Low,Moderate,Very High,sin venta
canal,,,,,
Moderno,6,12,8,2,22
Tradicional,3127,13355,7505,645,14554


,PDV con venta,mediana cajas/mes,p90,media,Gini
0,24660,2.850,8.621,4.104,0.441


## 4. Hallazgos y tabla de PDV para el 04, el 05 y el 06

In [5]:
hallazgos = pd.DataFrame([
    ("Datos del cliente", f"CP {Path(C.CLIENTE_CP).name}: {int(FUENTES_CLIENTE.filas.iloc[0]):,} PDV; AltScore: {', '.join(p.name for p in ALT)}. "
                          "Sin archivo de ventas: la venta es la venta media del CP.", "Huella en la hoja Datos del cliente."),
    ("Coordenadas", " · ".join(f"{k}: {v:,}" for k, v in coords.PDV.items()), "Se usan las que caen en un municipio del estado."),
    ("ZM", f"{len(cp):,} PDV en {C.ZM_NOMBRE}; {cp.con_venta.sum():,} con venta de {CAT.replace('CustomCat_', '')} ({cp.con_venta.mean():.0%}).",
     "Universo del 04, 05 y 06."),
    ("Canal", f"Moderno {(cp.canal == 'Moderno').sum():,} · Tradicional {(cp.canal == 'Tradicional').sum():,} (src/cadenas.py).",
     "Las letras y Golden Stores son del canal Tradicional."),
    ("Potencial", f"Sin venta: potencial vacío en {vacios.iloc[0]:.0f}% y clase vacía en {vacios.iloc[1]:.0f}% (el CP no lo calcula).",
     "En los entregables se dice el motivo (sin dato en el CP)."),
    ("Duplicados", f"{dup.sum():,} PDV comparten coordenadas exactas con otro.", "Se conservan (pueden ser locales del mismo predio)."),
], columns=["tema", "hallazgo", "implicación"])
with pd.option_context("display.max_colwidth", None):
    display(hallazgos)

salida = cp.set_index("pos_id").drop(columns=["pos_latitude", "pos_longitude"])
salida["cajas_mes_vida"] = salida["cajas_mes_activo"] = V.where(V > 0).to_numpy()     # sin ventas: la venta es la media del CP
for c_ in ["prefijos", "n_prefijos", "cajas_total", "meses_activos", "primera_venta", "ultima_venta", "meses_vida", "meses_span",
           "tasa_actividad", "recencia_meses", "estado_actividad"]:
    salida[c_] = np.nan
salida["alta_reciente"] = False
salida["z_robusto"] = eda.z_robusto(np.log1p(salida.cajas_mes_vida))
salida["atipico_volumen"] = salida.z_robusto.abs() > 3.5
salida.index.name = "pos_id_cp"
salida.reset_index().to_parquet(C.PROC / f"pdv_{C.CLIENTE}_{C.SLUG}.parquet", index=False)
out = C.OUT / f"00_validacion_cp_{C.CLIENTE}_{C.SLUG}.xlsx"
with pd.ExcelWriter(out) as xw:
    hallazgos.to_excel(xw, sheet_name="Hallazgos", index=False)
    FUENTES_CLIENTE.to_excel(xw, sheet_name="Datos del cliente", index=False)
    coords.to_excel(xw, sheet_name="Coordenadas")
    por_mun.round(2).to_excel(xw, sheet_name="Por municipio")
    por_sub.round(3).to_excel(xw, sheet_name="Por subcanal")
    clase.to_excel(xw, sheet_name="Clase CP")
print(f"Guardado: pdv_{C.CLIENTE}_{C.SLUG}.parquet ({len(salida):,} PDV, {int(salida.con_venta.sum()):,} con venta) | {out.name}")

,tema,hallazgo,implicación
0,Datos del cliente,"CP cp_arca_mex_sueros_202609.csv: 262,029 PDV; AltScore: geohex_geodig.parquet. Sin archivo de ventas: la venta es la venta media del CP.",Huella en la hoja Datos del cliente.
1,Coordenadas,"fuera de Jalisco: 202,842 · original: 59,187",Se usan las que caen en un municipio del estado.
2,ZM,"39,236 PDV en ZM Guadalajara; 24,660 con venta de sueros (63%).","Universo del 04, 05 y 06."
3,Canal,"Moderno 50 · Tradicional 39,186 (src/cadenas.py).",Las letras y Golden Stores son del canal Tradicional.
4,Potencial,Sin venta: potencial vacío en 100% y clase vacía en 100% (el CP no lo calcula).,En los entregables se dice el motivo (sin dato en el CP).
5,Duplicados,"5,833 PDV comparten coordenadas exactas con otro.",Se conservan (pueden ser locales del mismo predio).


Guardado: pdv_arca_zm_guadalajara.parquet (39,236 PDV, 24,660 con venta) | 00_validacion_cp_arca_zm_guadalajara.xlsx
